# FraudeCero · Grupo 6 · 04a · Modelo analítico (A3.2)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** entrenar un modelo de **clasificación** que detecte transacciones fraudulentas sin bloquear clientes legítimos (P3 · KPI3).

- **Partición declarada:** 80 % entrenamiento / 20 % prueba, **estratificada** por clase, semilla 42.
- **Línea base trivial:** un "modelo" que dice **siempre "normal"**. Tiene 99,8 % de exactitud y no detecta ningún fraude: demuestra por qué la exactitud no sirve aquí.
- **Modelos:** regresión logística y Random Forest (Spark MLlib) con **pesos por clase**.
- **Métricas:** AUC-PR (principal), AUC-ROC, precisión, recall y F1; además, análisis del **umbral de decisión**.
- **Datos:** `plata.transacciones_anonimizada` (sin id, montos ni tiempos exactos): el modelo respeta la privacidad desde el diseño.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "04a_modelo"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

datos = leer("plata", "transacciones_anonimizada")
features = [f"V{i}" for i in range(1, 29)] + ["log_amount", "hora"]
n0, n1 = datos.filter("Class = 0").count(), datos.filter("Class = 1").count()
ratio = n0 / n1
print(f"Normales: {n0:,} · Fraudes: {n1:,} · Peso del fraude: {ratio:.1f}")
datos = datos.withColumn("peso", F.when(F.col("Class") == 1, F.lit(ratio)).otherwise(F.lit(1.0)))

## 1. Partición entrenamiento / prueba (80/20 estratificada)

In [ ]:
t0 = time.time()
train = datos.sampleBy("Class", fractions={0: 0.8, 1: 0.8}, seed=42).cache()
test = datos.join(train.select("id_hash"), "id_hash", "left_anti").cache()
particion = pd.DataFrame([
    {"conjunto": "entrenamiento", "filas": train.count(), "fraudes": train.filter("Class=1").count()},
    {"conjunto": "prueba", "filas": test.count(), "fraudes": test.filter("Class=1").count()}])
particion["pct_fraude"] = (particion.fraudes / particion.filas * 100).round(3)
mostrar(particion)
registrar("modelo_particion", t0)

## 2. Entrenamiento

In [ ]:
t0 = time.time()
assembler = VectorAssembler(inputCols=features, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="Class", weightCol="peso", maxIter=50)
rf = RandomForestClassifier(featuresCol="features", labelCol="Class", weightCol="peso",
                            numTrees=100, maxDepth=8, seed=42)
modelos = {"Regresión logística": Pipeline(stages=[assembler, scaler, lr]).fit(train),
           "Random Forest": Pipeline(stages=[assembler, scaler, rf]).fit(train)}
registrar("modelo_entrenamiento", t0, filas=train.count())

## 3. Evaluación contra la línea base

In [ ]:
t0 = time.time()
ev_pr = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderPR")
ev_roc = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderROC")

def metricas(nombre, tp, fp, fn, tn, auc_pr, auc_roc):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"modelo": nombre, "AUC_PR": round(auc_pr, 4), "AUC_ROC": round(auc_roc, 4),
            "exactitud": round((tp + tn) / (tp + fp + fn + tn), 4), "precision": round(p, 4),
            "recall": round(r, 4), "F1": round(2 * p * r / (p + r), 4) if p + r else 0.0,
            "VP": tp, "FP": fp, "FN": fn, "VN": tn}

n_test = test.count(); f_test = test.filter("Class = 1").count()
filas = [metricas("Línea base (siempre normal)", 0, 0, f_test, n_test - f_test, f_test / n_test, 0.5)]
predicciones = {}
for nombre, modelo in modelos.items():
    pred = modelo.transform(test).withColumn("p_fraude", vector_to_array("probability")[1]).cache()
    predicciones[nombre] = pred
    c = pred.groupBy("Class", "prediction").count().toPandas()
    g = lambda y, yhat: int(c[(c.Class == y) & (c.prediction == yhat)]["count"].sum())
    filas.append(metricas(nombre, g(1, 1), g(0, 1), g(1, 0), g(0, 0), ev_pr.evaluate(pred), ev_roc.evaluate(pred)))

tabla_metricas = pd.DataFrame(filas)
tabla_metricas.to_csv(f"{EVID}/tablas/metricas_modelos.csv", index=False)
guardar(spark.createDataFrame(tabla_metricas), "oro", "metricas_modelo")
mostrar(tabla_metricas)
registrar("modelo_evaluacion", t0)
rf_m = tabla_metricas.set_index("modelo").loc["Random Forest"]
base = tabla_metricas.iloc[0]
print(f"📝 La línea base tiene {base.exactitud*100:.2f} % de exactitud pero recall 0: no detecta ningún fraude. "
      f"El Random Forest detecta {rf_m.VP} de {f_test} fraudes (recall {rf_m.recall:.3f}) con {rf_m.FP} falsas alarmas.")
print(f"📝 Su AUC-PR ({rf_m.AUC_PR:.3f}) es {rf_m.AUC_PR / base.AUC_PR:,.0f} veces la de la línea base ({base.AUC_PR:.4f}).")

## 4. Análisis del umbral de decisión (Random Forest)
El modelo entrega una probabilidad; el umbral decide desde qué probabilidad se marca fraude.

In [ ]:
t0 = time.time()
pr = predicciones["Random Forest"].select("Class", "p_fraude").toPandas()
filas_u = []
for u in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    yhat = (pr.p_fraude >= u).astype(int)
    tp = int(((yhat == 1) & (pr.Class == 1)).sum()); fp = int(((yhat == 1) & (pr.Class == 0)).sum())
    fn = int(((yhat == 0) & (pr.Class == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn)
    filas_u.append({"umbral": u, "VP": tp, "FP": fp, "FN": fn, "precision": round(p, 3),
                    "recall": round(r, 3), "F1": round(2 * p * r / (p + r), 3) if p + r else 0.0})
umbrales = pd.DataFrame(filas_u)
umbrales.to_csv(f"{EVID}/tablas/analisis_umbral.csv", index=False)
guardar(spark.createDataFrame(umbrales), "oro", "analisis_umbral")
mostrar(umbrales)
mejor = umbrales.sort_values("F1", ascending=False).iloc[0]
UMBRAL = float(mejor.umbral)
print(f"📝 El mejor equilibrio (F1 máx. = {mejor.F1:.3f}) está en el umbral {UMBRAL}: {mejor.VP} fraudes detectados con {mejor.FP} falsas alarmas.")
print("📝 Bajar el umbral sube el recall, pero por debajo de 0,4 cada fraude adicional cuesta decenas o cientos de falsas alarmas.")
registrar("modelo_umbral", t0)

## 5. Importancia de variables y persistencia en ORO

In [ ]:
t0 = time.time()
rf_model = modelos["Random Forest"].stages[-1]
imp = pd.DataFrame({"variable": features, "importancia": rf_model.featureImportances.toArray()}) \
        .sort_values("importancia", ascending=False)
guardar(spark.createDataFrame(imp), "oro", "importancia_variables")
mostrar(imp.head(10))
print(f"📝 Las 6 variables principales ({', '.join(imp.variable.head(6))}) explican {imp.importancia.head(6).sum()*100:.0f} % de la importancia.")

pred_oro = (predicciones["Random Forest"]
            .withColumn("prediccion_umbral", (F.col("p_fraude") >= UMBRAL).cast("int"))
            .select("id_hash", "dia", "hora", "franja_horaria", "rango_monto", "Class",
                    F.round("p_fraude", 4).alias("p_fraude"), "prediccion_umbral"))
guardar(pred_oro, "oro", "predicciones")
modelos["Random Forest"].write().overwrite().save(f"{LAKE}/modelos/random_forest")
with open(f"{EVID}/tablas/modelo_config.json", "w") as f:
    json.dump({"modelo": "RandomForestClassifier", "numTrees": 100, "maxDepth": 8, "seed": 42,
               "particion": "80/20 estratificada", "umbral": UMBRAL, "features": features}, f, indent=2)
registrar("modelo_persistencia", t0, df=pred_oro)

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)